# Clean Energy 2026 — WTX–CBO training (NASA)
**GitHub → Colab GPU → Google Drive checkpoints.** This notebook runs the current Python implementation; it does **not** establish reproduction of the article merely by completing epochs.

**Critical RUL rule:** Choose the absolute end-of-life capacity threshold (Ah) from the exact article/dataset protocol before running. The provisional value below is **not** a claim that the paper used it. Cells without a measured crossing are censored and the script intentionally refuses to train on a fabricated endpoint.

Place original NASA `B0005.mat`, `B0006.mat`, `B0007.mat`, `B0018.mat` in `MyDrive/NASA_original_source/mat/`. Check your sharing/IP requirements. Select **Runtime > Change runtime type > GPU** (optional, recommended).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys, pathlib, json
repo = '/content/wtx-cbo'
url = 'https://github.com/walidmchara/Hybrid-wavelet-transformer-XGBoost-framework-for-accurate-lithium-ion-battery-RUL-prediction.git'
if not pathlib.Path(repo, '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', url, repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo], check=True)
os.chdir(repo)
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

### Experiment settings
Change `EOL_CAPACITY_AH` to the **scientifically justified value**. This is a research precondition, not an automatically inferred target. For NASA the independent held-out battery is B0018. CBO is expensive and its search phase does **not** currently support resume; start with `RUN_CBO=False` for a fixed-parameter baseline.

In [ ]:
from pathlib import Path
NASA_DIR = Path('/content/drive/MyDrive/NASA_original_source/mat') # @param {type:"string"}
EOL_CAPACITY_AH = 1.4 # @param {type:"number"}
EPOCHS = 100 # @param {type:"integer"}
RUN_CBO = False # @param {type:"boolean"}
RUN_SMOKE_FIRST = True # @param {type:"boolean"}
SEED = 42 # @param {type:"integer"}
DRIVE_ROOT = Path('/content/drive/MyDrive/WTX_CBO_CleanEnergy_2026')
files = [NASA_DIR / f'{battery}.mat' for battery in ['B0005','B0006','B0007','B0018']]
missing = [str(p) for p in files if not p.is_file()]
if missing:
    raise FileNotFoundError('Place the original NASA .mat files in NASA_DIR: ' + repr(missing))
if EOL_CAPACITY_AH <= 0:
    raise ValueError('Select an explicit positive EOL capacity in Ah')
print('Selected EOL threshold:', EOL_CAPACITY_AH, 'Ah')
print('NASA files:', *files, sep='\n  ')
print('Results/checkpoints:', DRIVE_ROOT)

### Data preflight and unit tests
All four cells must have an observed threshold crossing. A censored cell is not a zero-RUL endpoint; stop and revise the study protocol instead of forcing a label.

In [ ]:
from wtx_cbo.core import load_nasa_battery_mat
from wtx_cbo.targets import eol_rul_from_capacity
for p in files:
    raw = load_nasa_battery_mat(p)
    labelled = eol_rul_from_capacity(raw, threshold_ah=EOL_CAPACITY_AH)
    print(p.stem, '| raw discharge cycles',len(raw),
          '| EOL cycle', int(labelled.iloc[-1]['cycle']),
          '| first capacity',float(raw.iloc[0]['capacity']),
          '| EOL capacity',float(labelled.iloc[-1]['capacity']))
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_eol_target_safety.py', 'tests/test_causal_preprocessing.py'], check=True)

### Train (checkpoints on Drive)
A 2-epoch smoke run is optional and goes to a **separate** output directory. The scientific baseline is the subsequent full run. Existing fixed-parameter checkpoints are resumed when present. After interruption rerun the notebook cells; check that the configuration and EOL threshold are unchanged.

In [ ]:
def run_training(epochs, output_name, *, cbo=False, resume=True):
    output_dir = DRIVE_ROOT / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = output_dir / 'recovery'
    cmd = [sys.executable, 'main.py', '--nasa-mat',
           *map(str,files),'--test-battery','B0018',
           '--eol-capacity-ah', str(EOL_CAPACITY_AH),
           '--lookback','32','--wavelet','db4','--wavelet-level','4',
           '--epochs',str(epochs),'--seed',str(SEED),
           '--checkpoint-dir',str(checkpoint_dir),'--output-dir',str(output_dir)]
    if cbo:
        cmd += ['--cbo','--cbo-population','4','--cbo-iterations','2',
                '--cbo-candidate-epochs','3']
    if resume and not cbo and (checkpoint_dir / 'latest.pt').exists():
        cmd.append('--resume')
    print('Training command:', ' '.join(cmd))
    subprocess.run(cmd, cwd=repo, check=True)
    return output_dir

if RUN_SMOKE_FIRST:
    smoke_output = run_training(2, 'smoke_2epochs', resume=False)
    print('Smoke test artifacts:', smoke_output)
full_output = run_training(EPOCHS,
                           'nasa_b0018_cbo_exploratory' if RUN_CBO else 'nasa_b0018_fixed',
                           cbo=RUN_CBO)
print('Finished run:', full_output)

### Inspect results
The script saves metrics, predictions, training history, model and residual booster. **Do not present these metrics as verified paper reproduction:** the new past-only wavelet and train-fitted IQR safeguards need empirical validation and the full article protocol still needs audit. CBO parameters in this notebook are deliberately exploratory, not paper-level exhaustive.

In [ ]:
import pandas as pd
meta_path = full_output / 'metadata.json'
print('Output files:', sorted(p.name for p in full_output.iterdir()))
if meta_path.is_file():
    print(json.dumps(json.loads(meta_path.read_text()), indent=2)[:5000])
for name in ['training_history.csv','test_predictions.csv','prediction_components.csv']:
    path = full_output / name
    if path.exists():
        print(name)
        display(pd.read_csv(path).tail())